In [0]:
password = dbutils.secrets.get('pawelnowak2004pri219_scope', 'pawel-neon-db-password')
spark.sql(f"""
CREATE CONNECTION IF NOT EXISTS neon_postgres_conn
TYPE POSTGRESQL
OPTIONS (
  host 'ep-xyz.aws.neon.tech',
  port '5432',
  user 'neondb_owner',
  password '{password}'
)
""")

In [0]:
%sql
CREATE FOREIGN CATALOG IF NOT EXISTS neon_catalog
USING CONNECTION neon_postgres_conn
OPTIONS (database 'neondb');

In [0]:
%sql
SELECT * FROM neon_catalog.public.artists_record_labels;

# Hybrid join: external vs delta tables

In [0]:
%sql
SELECT 
    ext.artist_name,
    ext.record_label,
    ext.country,
    ext.contract_signed_year,
    loc.title AS track_title,
    loc.album
FROM neon_catalog.public.artists_record_labels ext
INNER JOIN dbr_dev.music_analytics.dim_music_metadata loc
    ON lower(trim(ext.artist_name)) = lower(trim(loc.author))
ORDER BY ext.contract_signed_year ASC;

# Comparing the performance

In [0]:
%sql
-- 1. Federal query plan inspection (Predicate / Projection Pushdown to PostgreSQL)
EXPLAIN EXTENDED
SELECT artist_name, record_label 
FROM neon_catalog.public.artists_record_labels
WHERE contract_signed_year > 1985;

## delta table queries

In [0]:
%sql
-- Local delta plan inspection (Parquet Metadata Pruning)
EXPLAIN EXTENDED
SELECT author, album 
FROM dbr_dev.music_analytics.dim_music_metadata
WHERE author = 'Metallica';

# Discussion
# Lab 10 Notes: Lakehouse Federation & External Operational Databases

## Overview
In this lab, we connected Databricks directly to an external PostgreSQL database (hosted on Neon) using Lakehouse Federation. Instead of ingesting and copying raw files into cloud storage (our usual Bronze/Silver workflow), we queried the external transactional system in place and in real time.

---

## 1. Step-by-Step Implementation

1. **External Source Setup (Neon PostgreSQL):**
   - Spun up a serverless PostgreSQL instance on Neon.
   - Created a lookup reference table `artists_record_labels` and populated it with initial test records matching our YouTube project artists.

2. **Unity Catalog Connection Object:**
   - Registered a secure `CONNECTION` in Databricks SQL pointing to the Neon endpoint (host, port 5432, username, and password).
   - This object abstracts credentials securely—passwords remain hidden and are not hardcoded inside SQL queries.

3. **Foreign Catalog Creation:**
   - Created a `FOREIGN CATALOG` named `neon_catalog` referencing the connection object and the `neondb` database.
   - Unity Catalog automatically discovered remote schemas and mapped the PostgreSQL tables into the workspace catalog tree.

4. **Querying & Hybrid JOIN:**
   - Verified direct reads with a simple `SELECT` against `neon_catalog.public.artists_record_labels`.
   - Executed a hybrid `JOIN` combining the external PostgreSQL reference data with our local Delta table (`dim_music_metadata`).

---

## 2. Federation vs. Ingestion: Architectural Trade-Offs

The core engineering decision comes down to **querying in place (Federation)** vs. **copying data into the lakehouse (Ingestion into Delta Lake)**.

| Dimension | Lakehouse Federation (Query in Place) | Ingestion (Delta Lake / Medallion) |
| :--- | :--- | :--- |
| **Data Movement** | None. Data stays in the external database. | Full copy. Data is loaded and written to Parquet/Delta files. |
| **Data Freshness** | Immediate (zero latency). Always returns the live OLTP state. | Depends on pipeline frequency (e.g., hourly batch or near-real-time streaming). |
| **Analytical Speed** | Constrained by network bandwidth and the remote database CPU/RAM. | Extremely fast. Leverages Spark parallelism, file partitioning, and data skipping. |
| **Cost & Impact** | Low storage footprint, but generates outbound network traffic and adds load to the operational database. | Storage and compute costs for the pipeline, but queries never impact production OLTP systems. |
| **Best Used For** | Small lookup tables, reference data, ad-hoc audits, low-frequency queries. | High-volume fact tables, telemetry, core BI dashboards, ML model training. |

---

## 3. Under the Hood: Query Execution Plans

Looking at the `EXPLAIN` query plans highlights the structural differences:
- **Federated Queries:** Databricks relies on *Predicate & Projection Pushdown*. Filters (`WHERE`) and column selections (`SELECT`) are translated directly into native PostgreSQL queries and executed on the remote database. Only the necessary filtered rows travel over the network.
- **Delta Lake Queries:** Spark reads local Delta transaction logs and applies file-level *Data Skipping* (min/max statistics) directly on cloud storage.

---

## 4. Security & Network Considerations

- **Centralized Governance:** From an access management perspective, the external database behaves like any other Unity Catalog entity. We can grant table permissions (`GRANT SELECT`) to workspace users without provisioning accounts for them inside PostgreSQL.
- **Service Account Principle:** The `CONNECTION` object relies on a technical database user. This account should follow the principle of least privilege (read-only permissions) to avoid accidental writes to the operational database.
- **Operational Database Protection:** High query volume through federation risks connection pool exhaustion or CPU spikes on the source database. For high-concurrency dashboards, ingesting data into Bronze/Silver Delta tables remains the safer pattern.